# Preprocessing & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('Shape:', df.shape)
print('Duplicates:', df.duplicated().sum())

In [ ]:
# TotalCharges is object — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges:', df['TotalCharges'].isnull().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
# Drop customerID — not a feature
df.drop('customerID', axis=1, inplace=True)

# Encode binary Yes/No cols
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for c in binary_cols:
    df[c] = (df[c] == 'Yes').astype(int)

# Encode multi-value categoricals
cat_cols = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
            'StreamingMovies', 'Contract', 'PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

# SeniorCitizen is already 0/1

print('Shape after encoding:', df.shape)
df.head()

In [ ]:
# Split
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Scale numericals
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
X_train.head()

In [ ]:
# Feature importance comparison (Random Forest baseline, before/after engineering)
# 'Before' = just raw encoded features, 'After' = same here but we can check importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('Top 10 features:')
print(importances.head(10))

In [ ]:
# Evaluate baseline
y_pred = rf.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

In [ ]:
# Save processed data for next phase
import joblib
joblib.dump(scaler, '../models/scaler.pkl')

# Save splits
pd.DataFrame(X_train).to_parquet('../data/X_train.parquet')
pd.DataFrame(X_test).to_parquet('../data/X_test.parquet')
pd.DataFrame(y_train, columns=['Churn']).to_parquet('../data/y_train.parquet')
pd.DataFrame(y_test, columns=['Churn']).to_parquet('../data/y_test.parquet')

print('Saved scaler and processed splits to data/ and models/')